# FSL-Vision — Pipeline End-to-End (v2 final)
**Trabajo Final Integrador — Inteligencia Computacional (IC415)**

Extracción FathomNet **recortada al bounding box** y balanceada → AutoEDA → limpieza → **split estratificado + agrupado por imagen** (sin leakage) → Transfer Learning EfficientNet-B0 → fine-tuning → diagnóstico por clase. Extractor de características para la Fase 3 (anomalías).

> Borrá la carpeta `data/` previa en Drive antes de re-correr (no mezclar v1 con v2). Fases 2/2.5 requieren GPU (T4).

## Celda 1 — Preparación del Entorno

In [ ]:
# === CELDA 1: Preparación del Entorno ===

# 1) Montar Google Drive (acá persiste TODO: dataset, manifiesto y reporte).
from google.colab import drive
drive.mount('/content/drive')

# 2) Instalar dependencias.
#    - fathomnet: cliente oficial de FathomNet (MBARI)
#    - pyyaml: lectura del archivo de configuración
#    - ydata-profiling: motor de AutoEDA
#    - scikit-learn: split estratificado (Celda 6)
#    - Pillow: control de integridad y recortes (Celdas 5 y 7)
!pip install -q fathomnet pyyaml ydata-profiling scikit-learn Pillow

# NOTA COLAB: si la Celda 4 falla al importar ydata_profiling, reiniciá la
# sesión ("Entorno de ejecución -> Reiniciar sesión") y re-corré desde la 1.

# 3) Rutas centralizadas (evita inconsistencias entre celdas).
import os
PROJECT_DIR     = "/content/drive/MyDrive/IC415"          # <-- tu carpeta en Drive
DATA_RAW        = os.path.join(PROJECT_DIR, "data", "raw")
DATA_PROCESSED  = os.path.join(PROJECT_DIR, "data", "processed")
CONFIG_PATH     = os.path.join(PROJECT_DIR, "paraense_fauna.yaml")
MANIFIESTO      = os.path.join(DATA_RAW, "manifiesto_dataset.csv")
MANIFIESTO_LIMPIO = os.path.join(DATA_RAW, "manifiesto_limpio.csv")
REPORTE_EDA     = os.path.join(DATA_RAW, "reporte_eda.html")

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)
print("Proyecto:", PROJECT_DIR)
print("Crudo ->", DATA_RAW)
print("Procesado ->", DATA_PROCESSED)


## Celda 2 — Configuración v2 (paraense_fauna.yaml) — recorte a caja + más datos

In [ ]:
%%writefile /content/drive/MyDrive/IC415/paraense_fauna.yaml
# =====================================================================
# paraense_fauna.yaml  -  FSL-Vision Data Pipeline (v2: recorte a caja)
# Cambios clave respecto a v1, motivados por el diagnóstico (Celda 13):
#   - recorte_a_caja: aísla al animal y elimina el fondo de sedimento que
#     confundía a estrella/esponja/pulpo entre sí.
#   - solo_verificadas: false -> incorpora también anotaciones no verificadas,
#     multiplicando el volumen de las clases minoritarias (Octopoda pasa de ~35
#     a cientos de recortes disponibles).
#   - objetivo_por_clase: meta pareja para BALANCEAR el dataset.
# =====================================================================

descarga:
  proveedor_taxonomia: "fathomnet"   # expande cada concepto a sus descendientes
  solo_verificadas: false            # incluir NO verificadas -> volumen para minoritarias
  recorte_a_caja: true               # recortar al bounding box (quita el fondo confusor)
  objetivo_por_clase: 400            # meta de RECORTES por clase (balanceo)
  min_lado_caja_px: 32               # descartar cajas diminutas (recortes ilegibles)
  pausa_segundos: 0.15
  reintentos: 3

conceptos:
  - {carpeta: "asteroidea", concepto: "Asteroidea"}
  - {carpeta: "scyphozoa",  concepto: "Scyphozoa"}
  - {carpeta: "porifera",   concepto: "Porifera"}
  - {carpeta: "octopoda",   concepto: "Octopoda"}


## Celda 3 — Extracción FathomNet con recorte a Bounding Box + Manifiesto

In [ ]:
# === CELDA 3: Extracción FathomNet con RECORTE a Bounding Box + Manifiesto ===
import os, csv, time, requests, yaml
from io import BytesIO
from PIL import Image
from fathomnet.api import images, taxa
from fathomnet.dto import GeoImageConstraints

HEADERS = {"User-Agent": "IC415-UNaM-FSL-Vision/1.0"}
PLATAFORMAS = ["Tiburon", "Ventana", "Doc Ricketts", "i2MAP", "MiniROV", "Hercules"]


def plataforma_desde_url(url):
    if not url:
        return None
    u = url.replace(" ", "")
    for p in PLATAFORMAS:
        if p.replace(" ", "") in u:
            return p
    return None


def conjunto_descendientes(concepto, provider):
    """Concepto + todos sus descendientes taxonómicos. Sirve para quedarnos SOLO
    con las cajas de NUESTRA clase e ignorar las de otras especies que aparezcan
    en la misma imagen (p.ej. una esponja al lado de un pulpo)."""
    nombres = {t.name for t in taxa.find_taxa(provider, concepto)}
    nombres.add(concepto)
    return nombres


def bajar_imagen(url, reintentos, pausa):
    """Descarga robusta -> objeto PIL en RAM (no guardamos el frame entero en disco)."""
    for intento in range(reintentos):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return Image.open(BytesIO(r.content)).convert("RGB")
        except Exception as e:
            if intento == reintentos - 1:
                print(f"   ! descarga fallida ({e})")
                return None
            time.sleep(1.5 * (intento + 1))
    return None


def recortar(img_full, box, min_lado):
    """Recorta la región de la caja con clamp a los bordes. None si la caja es
    demasiado chica (recorte ilegible) o degenerada."""
    x, y, w, h = int(box.x), int(box.y), int(box.width), int(box.height)
    if w < min_lado or h < min_lado:
        return None
    x0, y0 = max(0, x), max(0, y)
    x1, y1 = min(img_full.width, x + w), min(img_full.height, y + h)
    if x1 <= x0 or y1 <= y0:
        return None
    return img_full.crop((x0, y0, x1, y1))


def fila_manifiesto(im, box_concept, clase, box, archivo):
    return {
        "uuid": im.uuid, "concepto": clase, "concepto_caja": box_concept,
        "url_origen": im.url, "profundidad_m": im.depthMeters,
        "temperatura_c": im.temperatureCelsius, "imaging_type": im.imagingType,
        "plataforma": plataforma_desde_url(im.url),
        "institucion": (im.contributorsEmail or "").split("@")[-1] or None,
        "box_w": int(box.width), "box_h": int(box.height),
        "archivo_local": archivo,
    }


def extraer_clase(concepto, carpeta, cfg, vistos):
    destino = os.path.join(DATA_RAW, carpeta)
    os.makedirs(destino, exist_ok=True)
    objetivo  = cfg["objetivo_por_clase"]
    provider  = cfg["proveedor_taxonomia"]
    solo_v    = cfg["solo_verificadas"]
    recortar_ = cfg["recorte_a_caja"]
    min_lado  = cfg["min_lado_caja_px"]
    pausa     = cfg["pausa_segundos"]
    reintentos = cfg["reintentos"]

    print(f"\n=== {carpeta} ({concepto}) ===")
    desc = conjunto_descendientes(concepto, provider); time.sleep(pausa)

    filas, n, offset, PAGINA = [], 0, 0, 200
    while n < objetivo:
        try:
            c = GeoImageConstraints(concept=concepto, taxaProviderName=provider,
                                    includeVerified=True, includeUnverified=not solo_v,
                                    limit=PAGINA, offset=offset)
            pagina = images.find(c); time.sleep(pausa)
        except Exception as e:
            print(f"   ! búsqueda fallida: {e}"); break
        if not pagina:
            print("   (no hay más imágenes disponibles)"); break

        for im in pagina:
            if n >= objetivo: break
            if not im.url: continue
            # SOLO las cajas que pertenecen a esta clase (concepto o descendiente).
            cajas = [b for b in (im.boundingBoxes or []) if b.concept in desc] if recortar_ \
                    else [None]
            if not cajas: continue

            # Idempotencia: nombres esperados; si ya están todos, no se descarga.
            if recortar_:
                esperados = [f"{im.uuid}_{i}.jpg" for i in range(len(cajas))]
            else:
                esperados = [f"{im.uuid}.jpg"]
            faltan = any(not os.path.exists(os.path.join(destino, e)) for e in esperados)
            img_full = bajar_imagen(im.url, reintentos, pausa) if faltan else None
            if faltan and img_full is None:
                continue

            for i, box in enumerate(cajas):
                if n >= objetivo: break
                archivo = esperados[i]
                ruta = os.path.join(destino, archivo)
                clave = f"{carpeta}/{archivo}"
                if clave in vistos: continue

                if not os.path.exists(ruta):
                    if recortar_:
                        crop = recortar(img_full, box, min_lado)
                        if crop is None: continue       # caja diminuta -> se descarta
                        crop.save(ruta)
                    else:
                        img_full.save(ruta)
                    time.sleep(pausa)

                vistos.add(clave)
                box_concept = box.concept if recortar_ else concepto
                box_ref = box if recortar_ else type("B", (), {"width": im.width or 0, "height": im.height or 0})()
                filas.append(fila_manifiesto(im, box_concept, concepto, box_ref,
                                             os.path.join(carpeta, archivo)))
                n += 1
                if n % 50 == 0:
                    print(f"   ... {n}/{objetivo}")
        offset += PAGINA
    print(f"   LISTO: {n} recortes")
    return filas


# ------------------------- Flujo principal -------------------------
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
cfg = config["descarga"]

campos = ["uuid", "concepto", "concepto_caja", "url_origen", "profundidad_m",
          "temperatura_c", "imaging_type", "plataforma", "institucion",
          "box_w", "box_h", "archivo_local"]

todas, vistos = [], set()
for c in config["conceptos"]:
    todas += extraer_clase(c["concepto"], c["carpeta"], cfg, vistos)

with open(MANIFIESTO, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=campos); w.writeheader(); w.writerows(todas)
print(f"\n==> Manifiesto: {MANIFIESTO}  ({len(todas)} recortes)")
print("   Recortes por clase:")
import collections
for cl, cnt in collections.Counter(r["concepto"] for r in todas).items():
    print(f"     {cl}: {cnt}")


## Celda 4 — AutoEDA

In [ ]:
# === CELDA 4: AutoEDA ===
import pandas as pd
from ydata_profiling import ProfileReport   # puede emitir un warning de deprecación: es inofensivo

df = pd.read_csv(MANIFIESTO)
print("Dimensiones del manifiesto:", df.shape)
print("\nImágenes por concepto:\n", df["concepto"].value_counts())

# Completitud de metadatos: clave para este dataset, ya que FathomNet
# no siempre trae profundidad/temperatura/plataforma.
print("\nCompletitud por columna (%):")
print((df.notna().mean() * 100).round(1))

perfil = ProfileReport(
    df,
    title="AutoEDA — Dataset de Normalidad FathomNet (FSL-Vision)",
    explorative=True,
)
perfil.to_file(REPORTE_EDA)
print(f"\n==> Reporte EDA guardado en: {REPORTE_EDA}")

# Para verlo embebido en el notebook (opcional):
# perfil.to_notebook_iframe()


## Celda 5 — Control de Integridad

In [ ]:
# === CELDA 5: Control de Integridad (limpieza de corruptos) ===
import os
import pandas as pd
from PIL import Image

def imagen_valida(ruta):
    """True si la imagen se abre y decodifica COMPLETA.
    - verify(): detecta cabeceras/estructura rotas.
    - load(): fuerza la decodificación real de los píxeles, lo que atrapa
      archivos TRUNCADOS (típico de descargas cortadas por desconexión en Colab).
    Se usan dos aperturas porque verify() deja el objeto inutilizable."""
    try:
        with Image.open(ruta) as im:
            im.verify()
        with Image.open(ruta) as im:
            im.load()
        return True
    except Exception:
        return False

df = pd.read_csv(MANIFIESTO)
n_inicial = len(df)

filas_ok, eliminadas = [], 0
for _, fila in df.iterrows():
    ruta = os.path.join(DATA_RAW, fila["archivo_local"])
    if os.path.exists(ruta) and imagen_valida(ruta):
        filas_ok.append(fila)
    else:
        # Corrupta o faltante: se borra del disco y se descarta del manifiesto.
        if os.path.exists(ruta):
            os.remove(ruta)
        eliminadas += 1

df_limpio = pd.DataFrame(filas_ok).reset_index(drop=True)
df_limpio.to_csv(MANIFIESTO_LIMPIO, index=False)

print(f"Imágenes evaluadas  : {n_inicial}")
print(f"Corruptas eliminadas: {eliminadas}")
print(f"Sobrevivientes      : {len(df_limpio)}")
print("\nSobrevivientes por clase:")
print(df_limpio["concepto"].value_counts())


## Celda 6 — Split sin Data Leakage — Estratificado + Agrupado por imagen (StratifiedGroupKFold)

In [ ]:
# === CELDA 6: Split sin Data Leakage — Estratificado + Agrupado por imagen ===
import os, shutil
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEMILLA: int = 42
N_SPLITS: int = 5     # 5 folds -> tomamos el primero como partición ~80/20

df = pd.read_csv(MANIFIESTO_LIMPIO)
df["clase"] = df["concepto"]      # el concepto de FathomNet es la etiqueta de clase
grupos = df["uuid"]               # AGRUPAR por imagen de origen (clave del anti-leakage)

# StratifiedGroupKFold combina DOS garantías al mismo tiempo:
#   - ESTRATIFICA: conserva las proporciones de clase en train y val (el desbalance
#     real se replica en ambos lados -> F1 de val representativo).
#   - AGRUPA: ningún 'uuid' (imagen) se reparte entre train y val.
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEMILLA)
train_idx, val_idx = next(sgkf.split(df, df["clase"], groups=grupos))
train_df = df.iloc[train_idx].copy()
val_df   = df.iloc[val_idx].copy()

# ======================================================================
# ¿POR QUÉ AGRUPAR POR 'uuid' Y NO HACER UN SPLIT AL AZAR?
# Con el recorte a caja, UNA imagen genera VARIOS recortes (un frame con 3 pulpos
# -> 3 muestras). Esos recortes comparten fondo, iluminación y a veces el mismo
# individuo: son casi-duplicados. Si un split al azar mandara unos a train y otros
# a val, el modelo estaría "viendo" en entrenamiento algo casi idéntico a lo que se
# evalúa -> métrica falsamente alta (DATA LEAKAGE por grupo). Agrupar por 'uuid'
# obliga a que TODOS los recortes de una misma foto caigan del mismo lado, cerrando
# esa fuga. La estratificación, en paralelo, preserva el balance de clases.
# ======================================================================

# Verificación dura: ninguna imagen puede estar en ambos lados (debe dar 0).
solapamiento = set(train_df["uuid"]) & set(val_df["uuid"])
print(f"Imágenes compartidas entre train y val (debe ser 0): {len(solapamiento)}")


def copiar_split(df_split, particion):
    """Copia físicamente cada recorte a data/processed/<particion>/<clase>/."""
    for _, fila in df_split.iterrows():
        destino_dir = os.path.join(DATA_PROCESSED, particion, fila["clase"])
        os.makedirs(destino_dir, exist_ok=True)
        origen  = os.path.join(DATA_RAW, fila["archivo_local"])
        destino = os.path.join(destino_dir, os.path.basename(fila["archivo_local"]))
        if os.path.exists(origen) and not os.path.exists(destino):
            shutil.copy2(origen, destino)   # copy2 conserva metadatos del archivo


copiar_split(train_df, "train")
copiar_split(val_df, "val")

print(f"\nTrain: {len(train_df)} recortes | Val: {len(val_df)} recortes\n")
resumen = pd.DataFrame({
    "train": train_df["clase"].value_counts(),
    "val":   val_df["clase"].value_counts(),
}).fillna(0).astype(int)
resumen["%_val"] = (resumen["val"] / (resumen["train"] + resumen["val"]) * 100).round(1)
print(resumen)


## Celda 7 — Sanity Check Visual

In [ ]:
# === CELDA 7: Sanity Check Visual (¿qué verá la red neuronal?) ===
import os, glob, random
import matplotlib.pyplot as plt
from PIL import Image

TARGET = 224   # tamaño de entrada típico de una CNN preentrenada (ImageNet)

def center_crop_cuadrado(im, size=TARGET):
    """Simula el preprocesamiento estándar tipo ImageNet:
    escala el lado más corto a `size` y recorta el CUADRADO central.
    Sirve para auditar el riesgo del recorte cuadrado: en frames panorámicos
    (16:9 de ROV) se descartan los bordes laterales, donde la fauna podría estar."""
    im = im.convert("RGB")
    w, h = im.size
    escala = size / min(w, h)
    im = im.resize((round(w * escala), round(h * escala)))
    w, h = im.size
    izq, arr = (w - size) // 2, (h - size) // 2
    return im.crop((izq, arr, izq + size, arr + size))

# Reunir todas las imágenes de train con su etiqueta (la carpeta = la clase).
rutas = []
train_dir = os.path.join(DATA_PROCESSED, "train")
for clase in sorted(os.listdir(train_dir)):
    for r in glob.glob(os.path.join(train_dir, clase, "*.jpg")):
        rutas.append((r, clase))

muestra = random.sample(rutas, min(16, len(rutas)))

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (ruta, clase) in zip(axes.ravel(), muestra):
    ax.imshow(center_crop_cuadrado(Image.open(ruta)))
    ax.set_title(clase, fontsize=10)
    ax.axis("off")
for ax in axes.ravel()[len(muestra):]:   # apagar ejes sobrantes
    ax.axis("off")
plt.suptitle(f"Sanity check — CenterCrop {TARGET}x{TARGET} sobre data/processed/train", fontsize=14)
plt.tight_layout()
plt.show()


## Celda 8 — Datasets y DataLoaders (PyTorch)

In [ ]:
# === CELDA 8: Datasets y DataLoaders (PyTorch) ===
import os
from collections import Counter
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from PIL import Image

# --- Hiperparámetros de datos ---
IMG_SIZE: int = 224
BATCH_SIZE: int = 32
MEAN = [0.485, 0.456, 0.406]   # media de ImageNet (obligatoria al usar pesos preentrenados)
STD  = [0.229, 0.224, 0.225]   # desvío de ImageNet


class Letterbox:
    """Transformación PIL -> PIL que lleva la imagen a `size` x `size` SIN deformar.

    Redimensiona conservando el aspect ratio (el lado MAYOR pasa a medir `size`) y
    rellena el sobrante con negro (padding). Resuelve el problema del CenterCrop,
    que recortaba la fauna pegada a los bordes: con Letterbox no se descarta NADA
    del frame y tampoco se estira la imagen (no distorsiona las proporciones del
    animal, algo crítico para que la CNN aprenda morfología real)."""

    def __init__(self, size: int = 224, fill: int = 0) -> None:
        self.size: int = size          # lado del cuadrado de salida
        self.fill: int = fill          # color del padding (0 = negro)

    def __call__(self, img: Image.Image) -> Image.Image:
        img = img.convert("RGB")
        w, h = img.size
        escala: float = self.size / max(w, h)                 # el lado mayor -> size
        nw, nh = max(1, round(w * escala)), max(1, round(h * escala))
        img = img.resize((nw, nh), Image.BILINEAR)
        lienzo = Image.new("RGB", (self.size, self.size), (self.fill,) * 3)
        lienzo.paste(img, ((self.size - nw) // 2, (self.size - nh) // 2))  # centrado
        return lienzo


# --- Pipelines de transformaciones ---
# TRAIN: Letterbox -> augmentation dinámica (geométrica + color) -> tensor -> Normalize.
# La augmentation es "online" (distinta en cada época): multiplica la variedad
# efectiva del dataset y frena el overfitting de las clases minoritarias, que el
# sampler va a mostrar muchas veces repetidas.
transform_train = transforms.Compose([
    Letterbox(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15, fill=0),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# VAL: evaluación determinista, SIN augmentation. Solo Letterbox + Normalize.
transform_val = transforms.Compose([
    Letterbox(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# --- Datasets desde data/processed/ (ImageFolder etiqueta según el nombre de carpeta) ---
train_dataset = datasets.ImageFolder(os.path.join(DATA_PROCESSED, "train"), transform=transform_train)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_PROCESSED, "val"),   transform=transform_val)
class_names = train_dataset.classes
num_classes: int = len(class_names)      # = 4 para tu dataset (se calcula, no se hardcodea)
print("Clases:", class_names, "| num_classes:", num_classes)

# --- WeightedRandomSampler: combate el desbalance 10:1 sobre-muestreando minorías ---
# Peso de cada clase = 1/frecuencia  ->  peso de cada muestra = peso de su clase.
# Con replacement=True, en cada época el flujo queda balanceado: Octopoda (35) se
# muestrea tan seguido como Porifera (358), SIN descartar datos (a diferencia del
# undersampling). Es la forma recomendada de atacar el desbalance sin perder señal.
targets = train_dataset.targets
conteos = Counter(targets)
peso_clase = {c: 1.0 / n for c, n in conteos.items()}
pesos_muestra = [peso_clase[t] for t in targets]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(pesos_muestra),
    num_samples=len(pesos_muestra),   # una época ve tantas muestras como el train original
    replacement=True,                 # imprescindible para poder repetir minorías
)
print("Conteo train por clase:", {class_names[c]: n for c, n in sorted(conteos.items())})

# --- DataLoaders ---
# IMPORTANTE: con sampler NO se pasa shuffle=True (el sampler ya define el orden).
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)}")


## Celda 9 — Modelo EfficientNet-B0 (Transfer Learning)

In [ ]:
# === CELDA 9: Modelo EfficientNet-B0 (Transfer Learning) ===
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Dispositivo: CUDA (GPU NVIDIA) > MPS (Apple Silicon) > CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Dispositivo:", device)

# Pesos preentrenados en ImageNet. Se usa la API moderna 'weights=' porque
# 'pretrained=True' está deprecado en torchvision reciente.
pesos = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=pesos)

# --- Transfer Learning: congelar el backbone, reentrenar solo el clasificador ---
# Con 543 imágenes y desbalance fuerte, reentrenar SOLO la cabeza:
#   (a) reduce drásticamente el costo de cómputo  -> UBICUIDAD (RA2),
#   (b) minimiza el overfitting (hay muy pocos datos para mover millones de pesos).
# Si querés embeddings adaptados al dominio marino para el detector de anomalías,
# poné FINE_TUNE_COMPLETO=True (y el LR baja solo a 1e-4 más abajo).
FINE_TUNE_COMPLETO: bool = False
if not FINE_TUNE_COMPLETO:
    for p in model.features.parameters():
        p.requires_grad = False

# Reemplazar la capa final (1280 -> num_classes), conservando el Dropout original.
in_features: int = model.classifier[1].in_features      # 1280 en B0
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

# Optimizador SOLO sobre parámetros entrenables. AdamW = Adam + weight decay
# desacoplado (regulariza mejor que Adam clásico).
params_entrenables = [p for p in model.parameters() if p.requires_grad]
LR: float = 1e-4 if FINE_TUNE_COMPLETO else 1e-3
optimizer = torch.optim.AdamW(params_entrenables, lr=LR, weight_decay=1e-4)

# CrossEntropyLoss SIN class weights: el desbalance ya lo maneja el
# WeightedRandomSampler. Sumarle pesos a la loss sería corregir DOS veces.
criterion = nn.CrossEntropyLoss()
print(f"Parámetros entrenables: {sum(p.numel() for p in params_entrenables):,}")


## Celda 10 — Entrenamiento y Validación (cabeza)

In [ ]:
# === CELDA 10: Entrenamiento y Validación ===
from sklearn.metrics import f1_score, accuracy_score

EPOCAS: int = 18
MODELO_PATH = os.path.join(PROJECT_DIR, "best_model_fase2.pth")


def correr_epoca(loader, entrenar: bool):
    """Una pasada completa por `loader`.
    Si entrenar=True, hace backprop y actualiza pesos. Devuelve
    (loss_promedio, accuracy, f1_macro) acumulando todas las predicciones."""
    model.train() if entrenar else model.eval()
    perdida_total: float = 0.0
    y_true, y_pred = [], []
    with torch.set_grad_enabled(entrenar):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if entrenar:
                optimizer.zero_grad()
            salidas = model(imgs)
            perdida = criterion(salidas, labels)
            if entrenar:
                perdida.backward()
                optimizer.step()
            perdida_total += perdida.item() * imgs.size(0)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(salidas.argmax(1).cpu().tolist())
    n = max(1, len(y_true))
    loss = perdida_total / n
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return loss, acc, f1


hist = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1"]}
mejor_f1: float = -1.0

for ep in range(1, EPOCAS + 1):
    tl, ta, tf = correr_epoca(train_loader, entrenar=True)
    vl, va, vf = correr_epoca(val_loader,   entrenar=False)
    for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1"],
                    [tl, vl, ta, va, tf, vf]):
        hist[k].append(v)
    print(f"Época {ep:02d}/{EPOCAS} | "
          f"train: loss {tl:.3f} acc {ta:.3f} f1 {tf:.3f} | "
          f"val: loss {vl:.3f} acc {va:.3f} f1 {vf:.3f}")

    # Se guarda el mejor modelo por F1 MACRO de validación, NO por accuracy:
    # con desbalance, accuracy premia predecir siempre la clase mayoritaria
    # (Porifera); el F1 macro promedia el desempeño por clase y castiga ignorar
    # a Octopoda/Scyphozoa.
    if vf > mejor_f1:
        mejor_f1 = vf
        torch.save(model.state_dict(), MODELO_PATH)
        print(f"   * nuevo mejor F1 val = {vf:.3f} -> guardado en {MODELO_PATH}")

print(f"\nEntrenamiento terminado. Mejor F1 macro (val): {mejor_f1:.3f}")


## Celda 11 — Visualización de Resultados (Fase 2)

In [ ]:
# === CELDA 11: Visualización de Resultados ===
import matplotlib.pyplot as plt

epocas = range(1, len(hist["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (clave, titulo) in zip(axes, [("loss", "Loss"), ("acc", "Accuracy"), ("f1", "F1-Score macro")]):
    ax.plot(epocas, hist[f"train_{clave}"], "o-", label="train")
    ax.plot(epocas, hist[f"val_{clave}"],   "s-", label="val")
    ax.set_title(titulo)
    ax.set_xlabel("Época")
    ax.set_ylabel(titulo)
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle("Fase 2 — Evolución del entrenamiento (EfficientNet-B0)", fontsize=14)
plt.tight_layout()
plt.show()


## Celda 12 — Fine-Tuning Completo del Backbone (Fase 2.5)

In [ ]:
# === CELDA 12: Fine-Tuning Completo del Backbone (Fase 2.5) ===
import matplotlib.pyplot as plt

MODELO_FT_PATH = os.path.join(PROJECT_DIR, "best_model_finetuned.pth")
EPOCAS_FT: int = 12      # ciclo corto: el backbone ya viene preentrenado + ajustado
PACIENCIA: int = 4       # early stopping: épocas seguidas sin mejorar antes de cortar

# 1) CARGA: traer los pesos de la Fase 2. Importante: el 'model' en memoria tras
#    la Celda 10 es el de la ÚLTIMA época, no el mejor. Recargar el checkpoint
#    garantiza que partimos del mejor estado (F1=0.727), no de uno cualquiera.
model.load_state_dict(torch.load(MODELO_PATH, map_location=device))
model = model.to(device)

# 2) DESCONGELAR TODO: ahora el backbone también recibe gradiente y aprende.
for p in model.parameters():
    p.requires_grad = True

# 3) DIFFERENTIAL LEARNING RATES (mitiga el Catastrophic Forgetting mejor que un
#    LR único). Las capas tempranas codifican rasgos universales (bordes, texturas)
#    que ya son buenos -> se mueven con un LR ínfimo. Las capas tardías y la cabeza,
#    más específicas, se adaptan algo más al dominio oceánico profundo.
bloques = list(model.features.children())                              # 9 bloques en B0
params_tempranos = [p for b in bloques[:5] for p in b.parameters()]    # genéricos  -> LR ínfimo
params_tardios   = [p for b in bloques[5:] for p in b.parameters()]    # específicos -> LR micro
params_cabeza    = list(model.classifier.parameters())                 # clasificador

optimizer = torch.optim.AdamW([
    {"params": params_tempranos, "lr": 1e-6},
    {"params": params_tardios,   "lr": 1e-5},
    {"params": params_cabeza,    "lr": 5e-5},
], weight_decay=1e-4)
# (Alternativa más conservadora: un único lr=1e-5 para todos los parámetros.)
print(f"Fine-tuning de {sum(p.numel() for p in model.parameters()):,} parámetros (todos entrenables)")

# 4) BASELINE: evaluamos el modelo cargado ANTES de tocar nada. Así sabemos si el
#    fine-tuning realmente raspa puntos sobre 0.727 o solo estabiliza la red.
_, _, f1_base = correr_epoca(val_loader, entrenar=False)
print(f"F1 macro val de partida (Fase 2): {f1_base:.3f}")
torch.save(model.state_dict(), MODELO_FT_PATH)   # el 'mejor' arranca siendo el baseline

# 5) CICLO con EARLY STOPPING por F1 macro de validación.
hist_ft = {k: [] for k in ["train_loss", "val_loss", "train_f1", "val_f1"]}
mejor_f1_ft = f1_base
sin_mejora = 0

for ep in range(1, EPOCAS_FT + 1):
    tl, ta, tf = correr_epoca(train_loader, entrenar=True)
    vl, va, vf = correr_epoca(val_loader,   entrenar=False)
    for k, v in zip(["train_loss", "val_loss", "train_f1", "val_f1"], [tl, vl, tf, vf]):
        hist_ft[k].append(v)
    print(f"FT época {ep:02d}/{EPOCAS_FT} | train loss {tl:.3f} f1 {tf:.3f} | "
          f"val loss {vl:.3f} f1 {vf:.3f}")

    if vf > mejor_f1_ft:
        mejor_f1_ft = vf
        sin_mejora = 0
        torch.save(model.state_dict(), MODELO_FT_PATH)
        print(f"   * mejora: F1 val {vf:.3f} -> guardado en {MODELO_FT_PATH}")
    else:
        sin_mejora += 1
        if sin_mejora >= PACIENCIA:
            print(f"   Early stopping: {PACIENCIA} épocas sin mejorar.")
            break

print(f"\nFine-tuning terminado. F1 base {f1_base:.3f} -> mejor F1 {mejor_f1_ft:.3f} "
      f"(delta {mejor_f1_ft - f1_base:+.3f})")

# 6) GRÁFICOS específicos de esta sub-fase.
ep_range = range(1, len(hist_ft["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(ep_range, hist_ft["train_loss"], "o-", label="train")
ax1.plot(ep_range, hist_ft["val_loss"],   "s-", label="val")
ax1.set_title("Fine-tuning — Loss"); ax1.set_xlabel("Época"); ax1.set_ylabel("Loss")
ax1.grid(alpha=0.3); ax1.legend()
ax2.plot(ep_range, hist_ft["train_f1"], "o-", label="train")
ax2.plot(ep_range, hist_ft["val_f1"],   "s-", label="val")
ax2.axhline(f1_base, ls="--", color="gray", label=f"F1 base Fase 2 ({f1_base:.3f})")
ax2.set_title("Fine-tuning — F1 macro"); ax2.set_xlabel("Época"); ax2.set_ylabel("F1 macro")
ax2.grid(alpha=0.3); ax2.legend()
plt.suptitle("Fase 2.5 — Fine-Tuning completo del backbone", fontsize=14)
plt.tight_layout(); plt.show()


## Celda 13 — Diagnóstico — Matriz de Confusión + F1 por Clase

In [ ]:
# === CELDA 13: Diagnóstico — Matriz de Confusión + F1 por Clase ===
# No entrena: solo INSTRUMENTA. Carga el mejor modelo y disecciona su desempeño
# clase por clase sobre validación, para ver dónde está realmente el problema
# (en vez de mirar un único F1 macro que promedia todo y esconde el detalle).
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)

# Modelo a diagnosticar. Por defecto el mejor (fine-tuned). Cambiá a MODELO_PATH
# para diagnosticar el de solo-cabeza (Fase 2) y comparar.
CHECKPOINT = MODELO_FT_PATH
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model = model.to(device)
model.eval()

# Inferencia determinista sobre validación (val_loader ya va SIN augmentation).
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        y_true.extend(labels.tolist())
        y_pred.extend(preds.tolist())

labels_idx = list(range(len(class_names)))

# --- Reporte por clase: precision, recall, F1 y SUPPORT (nº de imágenes en val) ---
# 'support' es clave acá: te recuerda con cuántas imágenes se midió cada F1.
print("Diagnóstico sobre:", os.path.basename(CHECKPOINT), "\n")
print(classification_report(y_true, y_pred, labels=labels_idx,
                            target_names=class_names, zero_division=0, digits=3))

# Clase más débil -> dónde conviene invertir (más datos, etc.).
f1_por_clase = f1_score(y_true, y_pred, average=None, labels=labels_idx, zero_division=0)
peor = class_names[int(np.argmin(f1_por_clase))]
print(f"Clase más débil: {peor}  (F1 = {f1_por_clase.min():.3f})")

# --- Matrices de confusión: cruda + normalizada por fila ---
# La normalizada por fila = RECALL por clase: de todas las imágenes que ERAN de
# la clase X, qué fracción se predijo bien (diagonal) y a dónde se fugaron las
# confusiones (fuera de la diagonal). Imprescindible con clases desbalanceadas,
# porque la matriz cruda está dominada visualmente por Porifera.
cm      = confusion_matrix(y_true, y_pred, labels=labels_idx)
cm_norm = confusion_matrix(y_true, y_pred, labels=labels_idx, normalize="true")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax1, cmap="Blues", colorbar=False)
ax1.set_title("Conteo absoluto")
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
    ax=ax2, cmap="Blues", colorbar=False, values_format=".2f")
ax2.set_title("Normalizada por fila (recall por clase)")
for ax in (ax1, ax2):
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.set_xticklabels(class_names, rotation=45, ha="right")
plt.suptitle(f"Diagnóstico del clasificador — {os.path.basename(CHECKPOINT)}", fontsize=14)
plt.tight_layout()
plt.show()
